In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
# Read the TSV files into the DataFrame
#Data Source Hasso Plattner Institut 
# NDPL - Non-duplicates
# DPL - Duplicates
# ncvoters - a snap shot of the snapshot: VR_Snapshot_20181106 
df_ncvoters = pd.read_csv('/Users/noimotbakare/Dropbox/Mac/Downloads/ncvoters.tsv', sep='\t')
DPL = pd.read_csv('/Users/noimotbakare/Dropbox/Erdös/Erdös_Project_SP2026/Fragmented_ID/ncvoters_DPL.tsv', sep='\t')
NDPL = pd.read_csv('/Users/noimotbakare/Dropbox/Erdös/Erdös_Project_SP2026/Fragmented_ID/ncvoters_NDPL.tsv', sep='\t')



In [ ]:
# Preprocessing for our fragemented ID analysis 
# Selecting identifyer variables not related to voting
df_ncvoters_frag_ID = df_ncvoters[[
#Stable Identifiers
#Only to be used for labeling /evaluation only 
# / not as a model feature
    'id', 'ncid', 'voter_reg_num', 
# Primary Model features - Strongest features, Strong entropy, Essential for matching
    #many variable such as name prefx and sufx are sparse we can either use none for missing or 0/1
    'first_name', 'midl_name', 'last_name', 'name_sufx_cd',
    #other varaiabes
# Adress Similarity features - Address is the second strongest identity anchor, 
    # Street name especially high discriminative signal # Unit numbers distinguish household
    #many variable such as unit designator are sparse we can either use none for missing or 0/1
    'house_num', 'street_name', 'street_dir', 'street_type_cd', 'unit_designator', 'unit_num', 'zip_code', 'res_city_desc',
#other varaiabes
# Demographic agreement indicators - Moderate/Supporting Features 
    # these will help us reduce false matches # they are agreement indicators, low-weight similarity features
    #age group rather than age because grouped/bin age is more stable
    'age', 'age_group', 'sex', 'race_code', 'race_desc', 'ethnic_code', 'ethnic_desc', 'birth_place',
# #other varaiabes 
 'phone_num','area_cd'   
 ]]
# print("\nSelected variables 'A' and 'C':")
print(df_ncvoters_frag_ID)

In [ ]:
# Adding labels to DPL and NDPL
DPL['label'] = 1 
NDPL['label'] = 0


print(DPL.head(10))
print(NDPL.head(10))

In [ ]:
#Merge Duplicate and Non Duplicate pairs
pairs = pd.concat([DPL, NDPL])

print(pairs.head(20))
print(pairs.tail(20))

In [ ]:
# merging ncvoters on DPL and NDPL
pairs = pairs.merge(
    df_ncvoters_frag_ID,
    left_on="id1",
    right_on="id",
    how="left"
)
print(pairs.info())
print(pairs.head(10))

In [ ]:
#merging id2 
pairs = pairs.merge(
    df_ncvoters_frag_ID,
    left_on="id2",
    right_on="id",
    how="left",
    suffixes=("_1", "_2")
)

print(pairs.info())
print(pairs.head(10))

Testing out different train test splits. 
1. Entity-Disjoint Split - splitting by "blocking key" or "entity clusters" to ensure train/test independence. Commonly used in entity linkage/duplicate detection. 

2. Stratified Split

1. Can our model identify duplicates among completely new people who happen to fall into the same blocking buckets?"


In [ ]:
# Get list of unique person IDs for splitting from ncvoters 
unique_ids = df_ncvoters['id'].unique()  

# Split persons into train (70%), validation (10%), test (20%) - we can adjust this 
train_ids, test_ids = train_test_split(unique_ids, test_size=0.2, random_state=42)
train_ids, val_ids = train_test_split(train_ids, test_size=0.125, random_state=42)  # 0.125 of 80% = 10% of total


print(f"Train IDs: {len(train_ids)}")
print(f"Val IDs: {len(val_ids)}")
print(f"Test IDs: {len(test_ids)}")


# Filter pairs based on id splits 
train_pairs = pairs[(pairs['id_1'].isin(train_ids)) & (pairs['id_2'].isin(train_ids))]
val_pairs = pairs[(pairs['id_1'].isin(val_ids)) & (pairs['id_2'].isin(val_ids))]
test_pairs = pairs[(pairs['id_1'].isin(test_ids)) & (pairs['id_2'].isin(test_ids))]

print(f"Train pairs: {len(train_pairs)}")
print(f"Val pairs: {len(val_pairs)}")
print(f"Test pairs: {len(test_pairs)}")

# X_train = train_pairs[siamese_features]
# y_train = train_pairs['label']


In [ ]:
# Check no ID overlap between sets
train_set = set(train_ids)
val_set = set(val_ids)
test_set = set(test_ids)

assert len(train_set & val_set) == 0, "Train and Val overlap!"
assert len(train_set & test_set) == 0, "Train and Test overlap!"
assert len(val_set & test_set) == 0, "Val and Test overlap!"

print("√ ID sets are disjoint")

# Check no person appears across pair sets
train_all_ids = set(train_pairs['id_1']).union(set(train_pairs['id_2']))
val_all_ids = set(val_pairs['id_1']).union(set(val_pairs['id_2']))
test_all_ids = set(test_pairs['id_1']).union(set(test_pairs['id_2']))

assert len(train_all_ids & val_all_ids) == 0, "ERROR: People appear in train and val!"
assert len(train_all_ids & test_all_ids) == 0, "ERROR: People appear in train and test!"
assert len(val_all_ids & test_all_ids) == 0, "ERROR: People appear in val and test!"

print("√ No person appears in multiple splits")

# Check label distribution
print("\nLabel distribution:")
print(f"Train: {train_pairs['label'].value_counts(normalize=True)}")
print(f"Val: {val_pairs['label'].value_counts(normalize=True)}")
print(f"Test: {test_pairs['label'].value_counts(normalize=True)}")

2. Stratification split - in this case we are asking "can our model classify pairs correctly, even if it's seen these people before in different pairs?"


In [ ]:
#Splitting off test
train_val_df, test_df = train_test_split(
    pairs,
    test_size=0.2,
    stratify=pairs["label"],
    random_state=42
)

# Splittting train and val 
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.125,   # 10% of original data
    stratify=train_val_df["label"],
    random_state=42
)

In [ ]:
#Size 
print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

In [ ]:
# Class distribution 
# Count
summary = pd.DataFrame({
    "Train": train_df["label"].value_counts(),
    "Validation": val_df["label"].value_counts(),
    "Test": test_df["label"].value_counts()
}).fillna(0).astype(int)

summary.loc["Total"] = summary.sum()

print(summary)

# Percentage
summary = pd.DataFrame({
    "Train(%)": train_df["label"].value_counts(normalize=True)*100,
    "Validation(%)": val_df["label"].value_counts(normalize=True)*100,
    "Test(%)": test_df["label"].value_counts(normalize=True)*100
}).fillna(0).astype(int)

summary.loc["Total"] = summary.sum()

print(summary)